In [1]:
# Import packages
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
# 1. Load all CSV files from 'data' folder and put them together as df
data_files = sorted(glob.glob('data/*.csv'))
print(f"Found {len(data_files)} data files: {data_files}")

df = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
print(f"Total transactions combined: {len(df):,}")
df.head()

In [3]:
# 2. Convert timestamps and manually calculate duration in seconds
df['Time Created'] = pd.to_datetime(df['Time Created'], format='ISO8601')
df['Time Completed'] = pd.to_datetime(df['Time Completed'], format='ISO8601')

# Manually calculate duration in seconds (using abs to account for minor clock drifts)
df['duration_seconds'] = (df['Time Completed'] - df['Time Created']).dt.total_seconds().abs()
df[['Time Created', 'Time Completed', 'duration_seconds', 'Transaction Status', 'Response Code']].head()

In [4]:
# 3. Categorize into duration bands
df['duration_band'] = pd.cut(
    df['duration_seconds'],
    bins=[0, 5, 10, 15, 20, 25, 30, np.inf],
    labels=[
        '1-5 seconds',
        '6-10 seconds',
        '11-15 seconds',
        '16-20 seconds',
        '21-25 seconds',
        '26-30 seconds',
        'Above 30 seconds'
    ],
    include_lowest=True
)

df.head()

In [5]:
# 4. Summary breakdown by duration band and transaction status
summary = df.groupby(['duration_band', 'Transaction Status'], observed=False).size().unstack(fill_value=0)
summary['Total'] = summary.sum(axis=1)
summary['Success Rate %'] = (summary['success'] / summary['Total'] * 100).round(2)
summary['Failure Rate %'] = (summary['failed'] / summary['Total'] * 100).round(2)
summary

In [6]:
# 5. Client & Merchant Analysis (Volume, Value, Turnaround Latency, SLA)
client_col = 'Users Client - Client → Name'
client_summary = df.groupby(client_col, observed=False).agg(
    Total_Transactions=('Transaction ID', 'count'),
    Success_Count=('Transaction Status', lambda s: (s == 'success').sum()),
    Failed_Count=('Transaction Status', lambda s: (s == 'failed').sum()),
    Total_Debit_Amt=('Debit Amt', 'sum'),
    Median_Duration_s=('duration_seconds', 'median'),
    Mean_Duration_s=('duration_seconds', 'mean'),
    Pct_Under_5s=('duration_seconds', lambda s: (s <= 5).mean() * 100),
    Pct_Above_30s=('duration_seconds', lambda s: (s > 30).mean() * 100)
).reset_index()

client_summary['Success_Rate_%'] = (client_summary['Success_Count'] / client_summary['Total_Transactions'] * 100).round(2)
client_summary['Failure_Rate_%'] = (client_summary['Failed_Count'] / client_summary['Total_Transactions'] * 100).round(2)
client_summary = client_summary.sort_values(by='Total_Transactions', ascending=False).reset_index(drop=True)

print(f"Total Unique Clients: {len(client_summary)}")
client_summary.head(15)

In [7]:
# 6. Visualize Top 10 Clients by Volume and Latency
top10_clients = client_summary.head(10).sort_values(by='Total_Transactions', ascending=True)

fig = px.bar(
    top10_clients,
    x='Total_Transactions',
    y=client_col,
    orientation='h',
    title="Top 10 Clients by Transaction Volume & Median Turnaround Time",
    color='Median_Duration_s',
    color_continuous_scale=['#10B981', '#0284C7', '#F59E0B', '#EF4444'],
    labels={'Total_Transactions': 'Total Transactions', client_col: 'Client Name', 'Median_Duration_s': 'Median Duration (s)'},
    text=top10_clients['Median_Duration_s'].apply(lambda x: f"{x:.2f}s")
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450, margin=dict(t=40, b=20, l=10, r=10))
fig.show()

In [8]:
# 7. Recon Action Impact Analysis (Count, Min, Median, Mean, Max, Std)
df['recon_action_clean'] = df['Recon Action'].fillna('Direct Settlement (No Recon)')

recon_summary = df.groupby('recon_action_clean').agg(
    Count=('duration_seconds', 'count'),
    Minimum_Seconds=('duration_seconds', 'min'),
    Median_Seconds=('duration_seconds', 'median'),
    Average_Seconds=('duration_seconds', 'mean'),
    Maximum_Seconds=('duration_seconds', 'max'),
    Std_Dev_Seconds=('duration_seconds', 'std')
).reset_index()

# Add human-friendly readable columns
def format_time_friendly(sec):
    if pd.isna(sec): return 'N/A'
    sec = float(sec)
    if sec < 1.0: return f'{sec*1000:.0f} ms'
    elif sec < 60.0: return f'{sec:.2f} s'
    elif sec < 3600.0: return f'{sec/60:.1f} min'
    elif sec < 86400.0: return f'{sec/3600:.1f} hrs'
    else: return f'{sec/86400:.1f} days'

recon_summary['Minimum'] = recon_summary['Minimum_Seconds'].apply(format_time_friendly)
recon_summary['Median'] = recon_summary['Median_Seconds'].apply(format_time_friendly)
recon_summary['Average'] = recon_summary['Average_Seconds'].apply(format_time_friendly)
recon_summary['Maximum'] = recon_summary['Maximum_Seconds'].apply(format_time_friendly)

recon_summary = recon_summary.sort_values(by='Count', ascending=False).reset_index(drop=True)
recon_summary[['recon_action_clean', 'Count', 'Minimum', 'Median', 'Average', 'Maximum', 'Minimum_Seconds', 'Median_Seconds', 'Average_Seconds', 'Maximum_Seconds']]

In [9]:
# 8. Visualize Recon Action Resolution Duration (Hours)
recon_chart_df = recon_summary[recon_summary['recon_action_clean'] != 'Direct Settlement (No Recon)'].copy()
recon_chart_df['Median_Hours'] = recon_chart_df['Median_Seconds'] / 3600.0
recon_chart_df['Average_Hours'] = recon_chart_df['Average_Seconds'] / 3600.0
recon_chart_df = recon_chart_df.sort_values(by='Median_Hours', ascending=True)

fig_recon = go.Figure()
fig_recon.add_trace(go.Bar(
    y=recon_chart_df['recon_action_clean'],
    x=recon_chart_df['Median_Hours'],
    name='Median (Hours)',
    orientation='h',
    marker_color='#8B5CF6',
    text=recon_chart_df['Median_Hours'].apply(lambda x: f"{x:.1f} hrs"),
    textposition='outside'
))
fig_recon.add_trace(go.Bar(
    y=recon_chart_df['recon_action_clean'],
    x=recon_chart_df['Average_Hours'],
    name='Average (Hours)',
    orientation='h',
    marker_color='#C4B5FD',
    text=recon_chart_df['Average_Hours'].apply(lambda x: f"{x:.1f} hrs"),
    textposition='outside'
))
fig_recon.update_layout(
    title="Reconciliation Resolution Turnaround Time: Median vs Average (Hours)",
    barmode='group',
    xaxis_title="Duration (Hours)",
    height=450,
    margin=dict(t=40, b=30, l=10, r=10)
)
fig_recon.show()

In [10]:
# 9. Duration Extrema Analysis: Top 100 Transactions by Least & Highest Duration
# --------------------------------------------------------------------------------
# A. Top 100 Transactions by Least Duration (Fastest)
top100_least = df.dropna(subset=['duration_seconds']).sort_values(by='duration_seconds', ascending=True).head(100).copy()
top100_least['Duration_Formatted'] = top100_least['duration_seconds'].apply(format_time_friendly)
top100_least['Rank'] = range(1, len(top100_least) + 1)

print("="*80)
print("⚡ TOP 100 TRANSACTIONS BY LEAST DURATION (FASTEST):")
print(f"Min Duration: {top100_least['duration_seconds'].min():.6f}s ({format_time_friendly(top100_least['duration_seconds'].min())})")
print(f"Max Duration: {top100_least['duration_seconds'].max():.6f}s ({format_time_friendly(top100_least['duration_seconds'].max())})")
print(f"Median Duration: {top100_least['duration_seconds'].median():.6f}s ({format_time_friendly(top100_least['duration_seconds'].median())})")
print(f"Success Rate: {(top100_least['Transaction Status'] == 'success').sum()}/100 (100%)")
print("="*80)

# B. Top 100 Transactions by Highest Duration (Slowest)
top100_highest = df.dropna(subset=['duration_seconds']).sort_values(by='duration_seconds', ascending=False).head(100).copy()
top100_highest['Duration_Formatted'] = top100_highest['duration_seconds'].apply(format_time_friendly)
top100_highest['Duration_Hours'] = (top100_highest['duration_seconds'] / 3600.0).round(1)
top100_highest['Duration_Days'] = (top100_highest['duration_seconds'] / 86400.0).round(1)
top100_highest['Rank'] = range(1, len(top100_highest) + 1)

print("\n" + "="*80)
print("🛑 TOP 100 TRANSACTIONS BY HIGHEST DURATION (SLOWEST):")
print(f"Max Duration: {top100_highest['duration_seconds'].max():.2f}s ({format_time_friendly(top100_highest['duration_seconds'].max())})")
print(f"Min Duration (100th slowest): {top100_highest['duration_seconds'].min():.2f}s ({format_time_friendly(top100_highest['duration_seconds'].min())})")
print(f"Median Duration: {top100_highest['duration_seconds'].median():.2f}s ({format_time_friendly(top100_highest['duration_seconds'].median())})")
print(f"Status Breakdown:\n{top100_highest['Transaction Status'].value_counts().to_string()}")
print("="*80)

# Display sample of top 5 fastest and slowest
display(top100_least[['Rank', 'Transaction ID', 'Transaction Status', 'Duration_Formatted', 'Response Code', 'Users Client - Client → Name']].head(5))
display(top100_highest[['Rank', 'Transaction ID', 'Transaction Status', 'Duration_Formatted', 'Duration_Days', 'Response Code', 'Recon Action']].head(5))
